In [11]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.tools import StructuredTool
from mcp import Client
import json

In [3]:
load_dotenv(override=True)

True

In [20]:
def convertir_outil(client, outil):
    async def appeler(**arguments) -> str:
        resultat = await client.call_tool(outil.name, arguments=arguments)
        if resultat.is_error:
            raise RuntimeError(f"Échec de {outil.name} : {resultat.content}")

        contenu = resultat.structured_content
        if contenu is None:
            contenu = [bloc.model_dump(mode="json") for bloc in resultat.content]
        return json.dumps(contenu, ensure_ascii=False)

    return StructuredTool.from_function(
        coroutine=appeler,
        name=outil.name,
        description=outil.description or outil.name,
        args_schema=outil.input_schema,
    )

In [21]:
# OPENAI_API_KEY doit être définie dans l’environnement.
URL = "http://localhost:24000/mcp"
async with Client(URL) as client:
    catalogue = await client.list_tools()
    outils = [convertir_outil(client, outil) for outil in catalogue.tools]

    agent = create_agent(
        model="openai:gpt-4.1-mini",
        tools=outils,
        system_prompt="Utilise les outils MCP disponibles pour répondre à la question. Répondre au format Markdown",
    )

    resultat = await agent.ainvoke({
        "messages": [{"role": "user", "content": "Nom et prénom et fonction et diplômes"}]
    })
    print(resultat["messages"][-1].content)

Voici les informations disponibles :

- Fonction : Professeur à l'ENSET Mohammedia, Université Hassan II de Casablanca
- Diplômes : 
  - Doctorat d'Etat (2015) dans le domaine des systèmes informatiques parallèles et distribués
  - Doctorat de troisième cycle (1996)
  - Diplôme de professeur second cycle (1993)

Je n'ai pas trouvé le nom et prénom précis. Souhaitez-vous que je recherche cela ?
